In [2]:
!pip install numpy matplotlib
!pip install scipy
import os
print(os.getcwd())


  Using cached numpy-2.1.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached matplotlib-3.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.3.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.54.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (163 kB)
  Using cached kiwisolver-1.4.7-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.3 kB)
  Using cached pillow-11.0.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.2.0-py3-none-any.whl.metadata (5.0 kB)
Using cached numpy-2.1.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.3 MB)
Using cached matplotlib-3.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
Using cached contourpy-1.3.0-cp311-cp311-manylinux_2_17_

In [19]:
# Including all pairwise distances without filtering out distances <1500 bp
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
from scipy.signal import fftconvolve

def read_data(file_path):
    """
    Reads the input file and extracts fragment coordinates for each chromosome.
    
    Parameters:
        file_path (str): Path to the input data file.
        
    Returns:
        dict: A dictionary where keys are chromosome names, and values are dictionaries mapping bin numbers to fragment counts.
    """
    data = {}
    with open(file_path, 'r') as file:
        header = next(file)  # Skip header
        for line in file:
            if "PASS" not in line:
                continue  # Only process lines with "PASS"
            parts = line.strip().split('\t')
            if len(parts) < 5:
                continue  # Ensure at least 5 columns are present
            list_of_frag_coord = parts[4]
            fragments = list_of_frag_coord.split(';')
            for frag in fragments:
                match = re.match(r'(chr\w+):(\d+)-(\d+)', frag)
                if match:
                    chr_, start, end = match.groups()
                    if chr_ not in data:
                        data[chr_] = {}
                    start, end = int(start), int(end)
                    bin_start = start // 500  # Bin numbering starts from 0
                    bin_end = end // 500
                    for bin_num in range(bin_start, bin_end + 1):
                        data[chr_][bin_num] = data[chr_].get(bin_num, 0) + 1
    return data

def convert_to_bin_counts(fragments_dict):
    """
    Converts the fragment count dictionary to a NumPy array indexed by bin number.
    
    Parameters:
        fragments_dict (dict): Dictionary mapping bin numbers to fragment counts.
        
    Returns:
        np.ndarray: An array where index represents bin number, and values are counts.
    """
    max_bin = max(fragments_dict.keys())
    bin_counts = np.zeros(max_bin + 1, dtype=int)  # Includes bin 0
    for bin_num, count in fragments_dict.items():
        bin_counts[bin_num] = count
    return bin_counts

def calculate_distance_histogram(bin_counts, max_distance_bins=None):
    """
    Computes the distance histogram using an autocorrelation function.
    
    Parameters:
        bin_counts (np.ndarray): Array of fragment counts for each bin.
        max_distance_bins (int, optional): Maximum distance (in bins).
        
    Returns:
        np.ndarray: Array representing the distance histogram.
    """
    autocorr = fftconvolve(bin_counts, bin_counts[::-1], mode='full')
    mid = len(autocorr) // 2
    distance_hist = autocorr[mid:]
    if max_distance_bins is not None:
        distance_hist = distance_hist[:max_distance_bins + 1]
    return distance_hist

def normalize_histogram_to_pmf(hist, bin_edges):
    """
    Normalizes the distance histogram to create a probability mass function (PMF).
    
    Parameters:
        hist (np.ndarray): Array representing the distance histogram.
        bin_edges (array-like): Bin edges for normalization.
        
    Returns:
        np.ndarray: Normalized PMF array.
    """
    if np.sum(hist) == 0:
        return np.zeros(len(bin_edges) - 1)
    
    distance_values = np.arange(len(hist))
    pmf, _ = np.histogram(distance_values, bins=bin_edges, weights=hist, density=False)
    pmf = pmf / np.sum(pmf)  # Normalize to ensure sum equals 1
    return pmf

def calculate_symmetric_kl_divergence(pmf1, pmf2):
    """
    Calculates the symmetric KL divergence between two PMFs.
    
    Parameters:
        pmf1 (np.ndarray): First PMF array.
        pmf2 (np.ndarray): Second PMF array.
        
    Returns:
        float: Symmetric KL divergence.
    """
    pmf1 = np.array(pmf1)
    pmf2 = np.array(pmf2)
    valid_bins = (pmf1 > 0) & (pmf2 > 0)
    if not np.any(valid_bins):
        return np.nan 
    pmf1_filtered = pmf1[valid_bins]
    pmf2_filtered = pmf2[valid_bins]
    pmf1_filtered /= pmf1_filtered.sum()
    pmf2_filtered /= pmf2_filtered.sum()
    kl_div_pq = entropy(pmf1_filtered, pmf2_filtered)
    kl_div_qp = entropy(pmf2_filtered, pmf1_filtered)
    symmetric_kl = 0.5 * (kl_div_pq + kl_div_qp)
    return symmetric_kl

def main(file_path, bin_size=500, distance_bins=50, min_prob=1e-5, max_distance=None):
    """
    Main function to process data, compute PMFs, and calculate symmetric KL divergence between chromosomes.
    
    Parameters:
        file_path (str): Path to the input data file.
        bin_size (int, optional): Size of each bin (default 500 bp).
        distance_bins (int, optional): Number of bins for PMF (initially set to 50).
        min_prob (float, optional): Minimum probability threshold for PMF bins (initially set to 1e-5).
        max_distance (int, optional): Maximum distance to consider
    """
    data = read_data(file_path)
    chromosome_pmf = {}
    
    # Determine the global maximum distance
    if max_distance is None:
        max_distance_possible_bins = 0
        for chrom, fragments_dict in data.items():
            bin_counts = convert_to_bin_counts(fragments_dict)
            distance_hist = calculate_distance_histogram(bin_counts)
            if len(distance_hist) > max_distance_possible_bins:
                max_distance_possible_bins = len(distance_hist)
        max_distance = max_distance_possible_bins * bin_size  
    
    # Compute the bin count based on max distance
    max_bin = max_distance // bin_size
    bin_edges = np.linspace(0, max_bin, distance_bins + 1)
    
    for chrom, fragments_dict in data.items():
        bin_counts = convert_to_bin_counts(fragments_dict)
        distance_hist = calculate_distance_histogram(bin_counts, max_distance_bins=max_bin)
        # No left truncation: including all distances
        pmf = normalize_histogram_to_pmf(distance_hist, bin_edges=bin_edges)
        pmf[pmf < min_prob] = 0
        if np.sum(pmf) > 0:
            pmf /= np.sum(pmf)
            chromosome_pmf[chrom] = pmf
    
    first_group = ["chr2L", "chr2R", "chr3L", "chr3R"]
    print("KL Divergence within group 2L, 2R, 3L, 3R:")
    for i in range(len(first_group)):
        for j in range(i + 1, len(first_group)):
            chr1, chr2 = first_group[i], first_group[j]
            if chr1 in chromosome_pmf and chr2 in chromosome_pmf:
                pmf1 = chromosome_pmf[chr1]
                pmf2 = chromosome_pmf[chr2]
                kl_divergence = calculate_symmetric_kl_divergence(pmf1, pmf2)
                print(f"KL divergence between {chr1} and {chr2}: {kl_divergence}")

    second_group = ["chrX", "chr4"]
    print("\nKL Divergence between 2L, 2R, 3L, 3R and X, 4:")
    for chrom_main in first_group:
        for chrom_other in second_group:
            if chrom_main in chromosome_pmf and chrom_other in chromosome_pmf:
                pmf_main = chromosome_pmf[chrom_main]
                pmf_other = chromosome_pmf[chrom_other]
                kl_divergence = calculate_symmetric_kl_divergence(pmf_main, pmf_other)
                print(f"KL divergence between {chrom_main} and {chrom_other}: {kl_divergence}")

if __name__ == "__main__":
    file_path = '/home/hzhou53/2024 Fall DNA and GIN model/GSM3347525NR_FDR_0.1_pseudoGEM_10000_enrichTest_master.txt'
    
    print("Running KL Divergence Calculation with all distances included:\n")
    main(file_path)


Running KL Divergence Calculation with all distances included:

KL Divergence within group 2L, 2R, 3L, 3R:
KL divergence between chr2L and chr2R: 0.05920000490242747
KL divergence between chr2L and chr3L: 0.012021762176092694
KL divergence between chr2L and chr3R: 0.03921620876434153
KL divergence between chr2R and chr3L: 0.07728872361472122
KL divergence between chr2R and chr3R: 0.15109697193321617
KL divergence between chr3L and chr3R: 0.048829563220129865

KL Divergence between 2L, 2R, 3L, 3R and X, 4:
KL divergence between chr2L and chrX: 0.06839954996646487
KL divergence between chr2L and chr4: 0.848757456546875
KL divergence between chr2R and chrX: 0.013729836071197413
KL divergence between chr2R and chr4: 0.9028297816182879
KL divergence between chr3L and chrX: 0.09622040321363139
KL divergence between chr3L and chr4: 0.6332355651805217
KL divergence between chr3R and chrX: 0.17544438047350358
KL divergence between chr3R and chr4: 0.9062710919969894


In [18]:
# Filtered out short pairwise distances (<1500 bp)
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
from scipy.signal import fftconvolve

def read_data(file_path):
    """
    Reads the input file and extracts fragment coordinates for each chromosome.
    
    Parameters:
        file_path (str): Path to the input data file.
        
    Returns:
        dict: A dictionary where keys are chromosome names, and values are dictionaries mapping bin numbers to fragment counts.
    """
    data = {}
    with open(file_path, 'r') as file:
        header = next(file)  # Skip header
        for line in file:
            if "PASS" not in line:
                continue  # Only process lines with "PASS"
            parts = line.strip().split('\t')
            if len(parts) < 5:
                continue  # Ensure at least 5 columns are present
            list_of_frag_coord = parts[4]
            fragments = list_of_frag_coord.split(';')
            for frag in fragments:
                match = re.match(r'(chr\w+):(\d+)-(\d+)', frag)
                if match:
                    chr_, start, end = match.groups()
                    if chr_ not in data:
                        data[chr_] = {}
                    start, end = int(start), int(end)
                    bin_start = start // 500  # Bin numbering starts from 0
                    bin_end = end // 500
                    for bin_num in range(bin_start, bin_end + 1):
                        data[chr_][bin_num] = data[chr_].get(bin_num, 0) + 1
    return data

def convert_to_bin_counts(fragments_dict):
    """
    Converts the fragment count dictionary to a NumPy array indexed by bin number.
    
    Parameters:
        fragments_dict (dict): Dictionary mapping bin numbers to fragment counts.
        
    Returns:
        np.ndarray: An array where index represents bin number, and values are counts.
    """
    max_bin = max(fragments_dict.keys())
    bin_counts = np.zeros(max_bin + 1, dtype=int)  # Includes bin 0
    for bin_num, count in fragments_dict.items():
        bin_counts[bin_num] = count
    return bin_counts

def calculate_distance_histogram(bin_counts, max_distance_bins=None):
    """
    Computes the distance histogram using an autocorrelation function.
    
    Parameters:
        bin_counts (np.ndarray): Array of fragment counts for each bin.
        max_distance_bins (int, optional): Maximum distance (in bins).
        
    Returns:
        np.ndarray: Array representing the distance histogram.
    """
    autocorr = fftconvolve(bin_counts, bin_counts[::-1], mode='full')
    mid = len(autocorr) // 2
    distance_hist = autocorr[mid:]
    if max_distance_bins is not None:
        distance_hist = distance_hist[:max_distance_bins + 1]
    return distance_hist

def normalize_histogram_to_pmf(hist, bin_edges, bin_start=0):
    """
    Normalizes the distance histogram to create a probability mass function (PMF).
    
    Parameters:
        hist (np.ndarray): Array representing the distance histogram.
        bin_edges (array-like): Bin edges for normalization.
        bin_start (int): Starting bin number (for labeling).
        
    Returns:
        np.ndarray: Normalized PMF array.
    """
    if np.sum(hist) == 0:
        return np.zeros(len(bin_edges) - 1)
    
    distance_values = bin_start + np.arange(len(hist))
    pmf, _ = np.histogram(distance_values, bins=bin_edges, weights=hist, density=False)
    pmf = pmf / np.sum(pmf)  
    return pmf

def calculate_symmetric_kl_divergence(pmf1, pmf2):
    """
    Calculates the symmetric KL divergence between two PMFs.
    
    Parameters:
        pmf1 (np.ndarray): First PMF array.
        pmf2 (np.ndarray): Second PMF array.
        
    Returns:
        float: Symmetric KL divergence.
    """
    pmf1 = np.array(pmf1)
    pmf2 = np.array(pmf2)
    valid_bins = (pmf1 > 0) & (pmf2 > 0)
    if not np.any(valid_bins):
        return np.nan 
    pmf1_filtered = pmf1[valid_bins]
    pmf2_filtered = pmf2[valid_bins]
    pmf1_filtered /= pmf1_filtered.sum()
    pmf2_filtered /= pmf2_filtered.sum()
    kl_div_pq = entropy(pmf1_filtered, pmf2_filtered)
    kl_div_qp = entropy(pmf2_filtered, pmf1_filtered)
    symmetric_kl = 0.5 * (kl_div_pq + kl_div_qp)
    return symmetric_kl

def main(file_path, bin_size=500, distance_bins=50, threshold_distance=1500, min_prob=1e-5, max_distance=None):
    """
    Main function to process data, compute PMFs, and calculate symmetric KL divergence between chromosomes.
    
    Parameters:
        file_path (str): Path to the input data file.
        bin_size (int, optional): Size of each bin (default 500 bp).
        distance_bins (int, optional): Number of bins for PMF (initially set to 50).
        threshold_distance (int, optional): Minimum distance to consider (initially set to 1500 bp).
        min_prob (float, optional): Minimum probability threshold for PMF bins (initially set to 1e-5).
        max_distance (int, optional): Maximum distance to consider
    """
    data = read_data(file_path)
    chromosome_pmf = {}
    threshold_bin = threshold_distance // bin_size  # 1500 / 500 = 3
    
    # Determine the global maximum distance
    if max_distance is None:
        max_distance_possible_bins = 0
        for chrom, fragments_dict in data.items():
            bin_counts = convert_to_bin_counts(fragments_dict)
            distance_hist = calculate_distance_histogram(bin_counts)
            if len(distance_hist) > max_distance_possible_bins:
                max_distance_possible_bins = len(distance_hist)
        max_distance = max_distance_possible_bins * bin_size  
    
    # Compute the bin count based on max distance
    max_bin = max_distance // bin_size
    bin_edges = np.linspace(threshold_bin, max_bin, distance_bins + 1)
    
    for chrom, fragments_dict in data.items():
        bin_counts = convert_to_bin_counts(fragments_dict)
        distance_hist = calculate_distance_histogram(bin_counts, max_distance_bins=max_bin)
        distance_hist_filtered = distance_hist[threshold_bin:]
        pmf = normalize_histogram_to_pmf(distance_hist_filtered, bin_edges=bin_edges, bin_start=threshold_bin)
        pmf[pmf < min_prob] = 0
        if np.sum(pmf) > 0:
            pmf /= np.sum(pmf)
            chromosome_pmf[chrom] = pmf
    
    first_group = ["chr2L", "chr2R", "chr3L", "chr3R"]
    print("KL Divergence within group 2L, 2R, 3L, 3R:")
    for i in range(len(first_group)):
        for j in range(i + 1, len(first_group)):
            chr1, chr2 = first_group[i], first_group[j]
            if chr1 in chromosome_pmf and chr2 in chromosome_pmf:
                pmf1 = chromosome_pmf[chr1]
                pmf2 = chromosome_pmf[chr2]
                kl_divergence = calculate_symmetric_kl_divergence(pmf1, pmf2)
                print(f"KL divergence between {chr1} and {chr2}: {kl_divergence}")

    second_group = ["chrX", "chr4"]
    print("\nKL Divergence between 2L, 2R, 3L, 3R and X, 4:")
    for chrom_main in first_group:
        for chrom_other in second_group:
            if chrom_main in chromosome_pmf and chrom_other in chromosome_pmf:
                pmf_main = chromosome_pmf[chrom_main]
                pmf_other = chromosome_pmf[chrom_other]
                kl_divergence = calculate_symmetric_kl_divergence(pmf_main, pmf_other)
                print(f"KL divergence between {chrom_main} and {chrom_other}: {kl_divergence}")

if __name__ == "__main__":
    file_path = '/home/hzhou53/2024 Fall DNA and GIN model/GSM3347525NR_FDR_0.1_pseudoGEM_10000_enrichTest_master.txt'
    
    print("Running KL Divergence Calculation after removing short distances:\n")
    main(file_path)


Running KL Divergence Calculation after removing short distances:

KL Divergence within group 2L, 2R, 3L, 3R:
KL divergence between chr2L and chr2R: 0.059261044371170385
KL divergence between chr2L and chr3L: 0.0119845639880839
KL divergence between chr2L and chr3R: 0.03928113963983904
KL divergence between chr2R and chr3L: 0.07738918110798498
KL divergence between chr2R and chr3R: 0.15122698466161852
KL divergence between chr3L and chr3R: 0.04880093890544896

KL Divergence between 2L, 2R, 3L, 3R and X, 4:
KL divergence between chr2L and chrX: 0.06851385988181455
KL divergence between chr2L and chr4: 0.8684490953099646
KL divergence between chr2R and chrX: 0.0137494015308769
KL divergence between chr2R and chr4: 0.9251205094814354
KL divergence between chr3L and chrX: 0.09636450179659617
KL divergence between chr3L and chr4: 0.6516796367362132
KL divergence between chr3R and chrX: 0.17568946477694367
KL divergence between chr3R and chr4: 0.9282422543522902
